# Notebook 103 — El agente RAG con MLflow Tracing

Tres cambios respecto al notebook 101:

| Cambio | Por qué importa |
|---|---|
| **Parametrizado** (`k`, `prompt_version`, `citation_policy`) | Sin parámetros no hay experimentos que comparar |
| **Instrumentado** con MLflow Tracing | Cuando el agente responde mal necesitas ver *dónde* falló |
| **Política de citación explícita** | Es la decisión que gobierna el balance precision/recall |

## Sobre el tracing

Un agente es una caja negra con varias etapas. Si la respuesta final es mala, ¿fue porque el
retrieval trajo basura, o porque el LLM ignoró buen contexto? Sin trazas, estás adivinando.

MLflow Tracing registra cada paso como un **span** anidado. Vas a poder abrir cualquier
respuesta en la UI y ver exactamente qué chunks entraron y qué prompt se envió.

**Este notebook se importa desde los notebooks 104, 105 y 106 mediante `%run`.** Por eso
define funciones y evita ejecutar trabajo pesado al importarse.

In [ ]:
%pip install -q -U databricks-ai-search "mlflow[databricks]>=3.1.0"
%restart_python

## 1. Configuración compartida

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA = "spark_examples"
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/agenteval"

T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_corpus"
T_TRAIN = f"{CATALOG}.{SCHEMA}.agenteval_train"
T_TEST = f"{CATALOG}.{SCHEMA}.agenteval_test"
T_DEV = f"{CATALOG}.{SCHEMA}.agenteval_train_dev"
T_HOLDOUT = f"{CATALOG}.{SCHEMA}.agenteval_train_holdout"

INDEX_NAME = f"{CATALOG}.{SCHEMA}.agenteval_corpus_index"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

### Experimento de MLflow

El experimento se crea bajo el directorio personal del usuario. Usar `/Shared` en un
workspace de clase provoca colisiones entre estudiantes.

In [ ]:
import mlflow

CURRENT_USER = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_PATH = f"/Users/{CURRENT_USER}/agenteval_rag"
RUTA_EXPERIMENTO = f"/Volumes/{CATALOGO}/{ESQUEMA}/agenteval_mlflow"
RUTA_DF_TEMPORAL = f"/Volumes/{CATALOGO}/{ESQUEMA}/agenteval_mlflow_tmp"
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.agenteval_mlflow_tmp
""")
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.agenteval_mlflow
""")
# MLflow usará este directorio al guardar modelos Spark ML
os.environ["MLFLOW_DFS_TMP"] = RUTA_EXPERIMENTO
os.environ["SPARKML_TEMP_DFS_PATH"]=RUTA_DF_TEMPORAL

mlflow.set_experiment(EXPERIMENT_PATH)
print(f"Experimento: {EXPERIMENT_PATH}")
print(f"MLflow     : {mlflow.__version__}")

### Conexión al índice y al LLM

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.ai_search.client import VectorSearchClient

PREFERRED_LLMS = [
    "databricks-meta-llama-3-1-8b-instruct"
]

w = WorkspaceClient()
available = {e.name for e in w.serving_endpoints.list()}
LLM_ENDPOINT = next((m for m in PREFERRED_LLMS if m in available), None)
assert LLM_ENDPOINT, (
    f"Ningún modelo preferido disponible.\nPreferidos: {PREFERRED_LLMS}\n"
    f"Disponibles: {sorted(available)}"
)

openai_client = w.serving_endpoints.get_open_ai_client()

vsc = VectorSearchClient(disable_notice=True)
VS_ENDPOINT = vsc.list_endpoints().get("endpoints", [])[0]["name"]
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)

print(f"LLM      : {LLM_ENDPOINT}")
print(f"Endpoint : {VS_ENDPOINT}")
print(f"Índice   : {INDEX_NAME}")

## 2. Los prompts

Dos versiones. La diferencia entre ambas es una sola instrucción, y es la hipótesis central
del experimento que corres en el notebook 105.

- **v1** — pide citar los chunks usados. Sin más.
- **v2** — pide explícitamente el **conjunto mínimo** de chunks, e instruye a no citar un
  chunk solo porque parezca relacionado.

¿Por qué debería importar? Porque `gold_chunk_ids` en el dataset es, por definición, el
conjunto mínimo de evidencia. Un agente que cita todo lo que le suena relevante tiene recall
alto y precisión baja. La v2 apunta a la precisión sin sacrificar el recall.

In [ ]:
PROMPTS = {
    "v1": """You are a grounded QA assistant. Answer ONLY using the provided chunks.

Rules:
1. If the chunks do not contain the answer, reply exactly: NOT_FOUND
2. Answer in one short sentence, copying key facts verbatim from the chunks.
3. End your reply with a line in exactly this format:
CITED: chunk_id_1, chunk_id_2""",

    "v2": """You are a grounded QA assistant. Answer ONLY using the provided chunks.

Rules:
1. If the chunks do not contain the answer, reply exactly: NOT_FOUND
2. Answer in one short sentence, copying key facts verbatim from the chunks.
3. Cite the MINIMAL set of chunks required to support your answer. Most questions need
   only ONE chunk. Do NOT cite a chunk merely because it looks related to the topic:
   cite it only if your answer would be unsupported without it.
4. End your reply with a line in exactly this format:
CITED: chunk_id_1""",
}

CITATION_POLICIES = ["model", "intersect", "all_retrieved", "top1"]

## 3. El retriever, instrumentado

El span de tipo `RETRIEVER` no es una etiqueta decorativa. MLflow espera un **formato de
salida específico** — una lista de objetos `Document` con `page_content` y `metadata` — y ese
formato es lo que permite dos cosas:

1. Que la UI renderice los documentos recuperados de forma legible
2. Que los scorers `RetrievalGroundedness` y `RetrievalRelevance` del notebook 104 encuentren
   automáticamente el contexto, sin que tengas que pasárselo a mano

Si el span no cumple el esquema, esos scorers no tienen de dónde leer y fallan.

In [ ]:
from mlflow.entities import Document, SpanType

SEARCH_COLUMNS = ["chunk_id", "doc_id", "title", "doc_type", "chunk_text"]


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve(question: str, k: int = 3) -> list[dict]:
    """Recupera los k chunks más relevantes del índice de AI Search."""
    res = index.similarity_search(
        query_text=question, columns=SEARCH_COLUMNS, num_results=k
    )
    rows = res.get("result", {}).get("data_array", [])
    cols = SEARCH_COLUMNS + ["score"]
    chunks = [dict(zip(cols, r)) for r in rows]

    # Formato que MLflow exige para spans RETRIEVER
    span = mlflow.get_current_active_span()
    if span is not None:
        span.set_outputs([
            Document(
                id=c["chunk_id"],
                page_content=c["chunk_text"],
                metadata={
                    "doc_uri": c["chunk_id"],
                    "chunk_id": c["chunk_id"],
                    "doc_id": c["doc_id"],
                    "title": c["title"],
                    "doc_type": c["doc_type"],
                    "relevance_score": c["score"],
                },
            )
            for c in chunks
        ])

    return chunks

## 4. El generador, instrumentado

In [ ]:
import re
import time


@mlflow.trace(span_type=SpanType.LLM)
def generate(question: str, chunks: list[dict], prompt_version: str = "v1") -> tuple[str, list[str]]:
    """Genera la respuesta y extrae las citas declaradas por el modelo."""
    if not chunks:
        return "NOT_FOUND", []

    blocks = [f"[{c['chunk_id']}] {c['chunk_text']}" for c in chunks]
    user_msg = "CHUNKS:\n" + "\n\n".join(blocks) + f"\n\nQUESTION: {question}"

    raw = None
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=LLM_ENDPOINT,
                messages=[
                    {"role": "system", "content": PROMPTS[prompt_version]},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=300,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

    valid_ids = {c["chunk_id"] for c in chunks}
    match = re.search(r"CITED:\s*(.+?)\s*$", raw, flags=re.MULTILINE | re.DOTALL)

    if match:
        cited = [c.strip() for c in match.group(1).replace("\n", ",").split(",") if c.strip()]
        cited = [c for c in cited if c in valid_ids]
        answer = raw[: match.start()].strip()
    else:
        cited = [c["chunk_id"] for c in chunks]
        answer = raw.strip()

    return answer, cited

## 5. Políticas de citación

Esta es la pieza de diseño más interesante del agente. El modelo propone citas; la política
decide qué se hace con esa propuesta:

| Política | Comportamiento | Efecto esperado |
|---|---|---|
| `model` | Confía en lo que declaró el LLM | Depende de la calidad del prompt |
| `intersect` | Solo citas que además fueron recuperadas | Elimina IDs inventados |
| `all_retrieved` | Cita los `k` chunks recuperados | Recall máximo, precisión mínima |
| `top1` | Solo el chunk mejor rankeado | Precisión alta, recall bajo en preguntas multi-chunk |

Separar el retrieval de la citación permite algo importante: **recuperar amplio para darle
contexto al modelo, pero citar estrecho para no diluir la precisión.**

In [ ]:
def apply_citation_policy(cited: list[str], chunks: list[dict], policy: str) -> list[str]:
    """Aplica la política de citación sobre las citas propuestas por el modelo."""
    retrieved_ids = [c["chunk_id"] for c in chunks]

    if policy == "all_retrieved":
        return retrieved_ids
    if policy == "top1":
        return retrieved_ids[:1]
    if policy == "intersect":
        return [c for c in cited if c in set(retrieved_ids)]
    if policy == "model":
        return cited
    raise ValueError(f"Política desconocida: {policy}. Opciones: {CITATION_POLICIES}")

## 6. El agente completo

In [ ]:
@mlflow.trace(name="rag_agent")
def rag_agent(
    question: str,
    k: int = 3,
    prompt_version: str = "v1",
    citation_policy: str = "model",
) -> dict:
    """Agente RAG: retrieve -> generate -> cite.

    Devuelve dict con answer, cited_chunk_ids y retrieved_chunk_ids.
    """
    chunks = retrieve(question, k)
    answer, cited = generate(question, chunks, prompt_version)
    final_citations = apply_citation_policy(cited, chunks, citation_policy)

    return {
        "answer": answer,
        "cited_chunk_ids": final_citations,
        "retrieved_chunk_ids": [c["chunk_id"] for c in chunks],
    }

## 7. Prueba del agente

Esta sección se salta cuando el notebook se importa con `%run` desde otro notebook: no tiene
sentido gastar llamadas al LLM cada vez que 104 o 105 importan las funciones.

In [ ]:
def _is_imported() -> bool:
    """True si este notebook fue invocado vía %run desde otro."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return "103_rag_agent" not in ctx.notebookPath().get()
    except Exception:
        return False


RUNNING_STANDALONE = not _is_imported()
print(f"Modo: {'ejecución directa' if RUNNING_STANDALONE else 'importado vía %run'}")

In [ ]:
if RUNNING_STANDALONE:
    row = spark.table(T_DEV).orderBy("question_id").first()
    question, gold = row["question"], list(row["gold_chunk_ids"])

    print(f"Pregunta : {question}")
    print(f"Gold     : {gold}\n")

    for policy in ["model", "all_retrieved", "top1"]:
        out = rag_agent(question, k=3, prompt_version="v2", citation_policy=policy)
        acierto = "correcto" if set(out["cited_chunk_ids"]) == set(gold) else "no coincide"
        print(f"  política={policy:<14} citas={out['cited_chunk_ids']}  ({acierto})")
        time.sleep(0.5)

    print(f"\nRespuesta generada: {out['answer']}")

## 8. Inspeccionar las trazas en la UI

Este es el momento central del notebook. Acabas de generar trazas; ahora míralas.

**Cómo llegar:**

1. Menú lateral izquierdo → **Experiments**
2. Abre el experimento `agenteval_rag`
3. Pestaña **Traces**
4. Haz clic en cualquier traza de la lista

**Qué buscar:**

- El árbol de spans a la izquierda: `rag_agent` como raíz, con `retrieve` y `generate`
  anidados debajo
- Clic en `retrieve` → los documentos recuperados, renderizados como tarjetas legibles
  (ese renderizado es consecuencia directa de haber respetado el esquema de `Document`)
- Clic en `generate` → el prompt completo enviado y la respuesta cruda del modelo
- La latencia de cada span: verás que el retrieval es rápido y el LLM domina el tiempo total

**Por qué esto importa:** cuando un agente responde mal en producción, esta vista te dice en
segundos si el problema fue el retrieval o la generación. Sin ella, dependes de reproducir el
error a mano.

In [ ]:
if RUNNING_STANDALONE:
    last_id = mlflow.get_last_active_trace_id()
    if last_id:
        trace = mlflow.get_trace(last_id)
        print(f"Traza: {last_id}\n")
        print(f"{'SPAN':<24} {'TIPO':<14} {'ms':>8}")
        print("-" * 48)
        for s in trace.data.spans:
            ms = (s.end_time_ns - s.start_time_ns) / 1e6
            print(f"{s.name:<24} {str(s.span_type):<14} {ms:>8.0f}")

        retr = trace.search_spans(span_type=SpanType.RETRIEVER)
        if retr and retr[0].outputs:
            print(f"\nDocumentos en el span RETRIEVER: {len(retr[0].outputs)}")
            for d in retr[0].outputs:
                meta = d.get("metadata", {}) if isinstance(d, dict) else {}
                content = d.get("page_content", "") if isinstance(d, dict) else ""
                print(f"  - {meta.get('chunk_id')}: {content[:70]}...")

## 9. Nota de diseño: por qué no desplegamos el agente

MLflow permite registrar un agente como modelo (`ChatAgent` / `ResponsesAgent`) y desplegarlo
como endpoint de Model Serving. Aquí **no** lo hacemos, por dos razones:

1. **Free Edition limita los endpoints de Model Serving activos.** Ya estás consumiendo la
   única unidad de AI Search; añadir un endpoint propio compite por cuota.
2. **No aporta al objetivo.** Todo lo que evalúas en 104–106 funciona igual con el agente
   como función Python trazada. El despliegue es un tema de operación, no de calidad.

En un proyecto real con edición de pago, el paso siguiente sería `mlflow.pyfunc.log_model()`
con la interfaz `ResponsesAgent` y despliegue con `agents.deploy()`. La lógica del agente no
cambiaría — solo se envuelve.